In [ ]:
# ============================
# STEP 1: SETUP & IMPORTS
# ============================

# 1️⃣ Clone Ultralytics (YOLOv8/YOLO11) repo
!git clone https://github.com/ultralytics/ultralytics
%cd /content/ultralytics

# 2️⃣ Install in editable mode so our code changes take effect
!pip install -e .

# 3️⃣ All imports we will need later (import everything in Step 1)
import os
import re
import glob
import shutil
from pathlib import Path

import torch
import yaml

print("✅ Environment ready!")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("⚠ No GPU detected.")

Cloning into 'ultralytics'...
remote: Enumerating objects: 75957, done.
remote: Counting objects: 100% (255/255), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 75957 (delta 211), reused 138 (delta 138), pack-reused 75702 (from 3)
Receiving objects: 100% (75957/75957), 40.64 MiB | 21.23 MiB/s, done.
Resolving deltas: 100% (57054/57054), done.
/content/ultralytics
Obtaining file:///content/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.234-0.editable-py3-none-any.whl size=23169 sha256=85e98195449e518af0bb7b4461a8cc65fd3111a6ea5b68c7d78d85415cec30a6
  Stored in directory: /tmp/pip-ephem-wheel-cache-7syq1ddu/wheels/60/e0/59/e2f034f296abbdca5c21e3f5be76b9ca685f13c7bd17f8b58c
Suc

In [ ]:
# STEP 2A: Configure Kaggle API (run this once)

from google.colab import files
import os

print("📁 Please upload your kaggle.json file (from Kaggle account settings).")
uploaded = files.upload()  # <-- You select kaggle.json here

if 'kaggle.json' not in uploaded:
    raise FileNotFoundError("❌ kaggle.json not uploaded. Please try again.")

# Create kaggle folder and move the file
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/

# Set proper permissions so Kaggle CLI accepts it
!chmod 600 /root/.kaggle/kaggle.json

print("✅ Kaggle API configured successfully!")

📁 Please upload your kaggle.json file (from Kaggle account settings).


Saving kaggle.json to kaggle.json
✅ Kaggle API configured successfully!


In [ ]:
import torch
import torch.nn as nn

class GlobalSelfAttention(nn.Module):
    def __init__(self, in_channels):
        super(GlobalSelfAttention, self).__init__()
        self.in_channels = in_channels

        # Branch 1: Standard Conv path
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        # Branch 2: Simpler Conv path
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        # Final projection layer to combine branches and generate attention map
        self.final_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.sigmoid = nn.Sigmoid()

        print(f"Initialized GlobalSelfAttention with in_channels={in_channels}")

    def forward(self, x):
        # Apply parallel branches
        out1 = self.branch1(x)
        out2 = self.branch2(x)

        # Sum the outputs of the parallel branches
        combined_features = out1 + out2

        # Generate attention map
        attention_map = self.sigmoid(self.final_conv(combined_features))

        # Apply attention to the input feature map
        out = x * attention_map

        return out

print("✅ GlobalSelfAttention class defined.")

✅ GlobalSelfAttention class defined.


In [ ]:
import torch
import torch.nn as nn
from ultralytics.nn.modules import Conv, C2f # Assuming C2f is still needed for structure

# Re-using the ASR_Attention definition for clarity and avoiding re-execution errors
class GlobalSelfAttention(nn.Module):
    def __init__(self, in_channels):
        super(GlobalSelfAttention, self).__init__()
        self.in_channels = in_channels

        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.final_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        combined_features = out1 + out2
        attention_map = self.sigmoid(self.final_conv(combined_features))
        out = x * attention_map
        return out


class C2f_ASR(nn.Module):
    """C2f block with Attention-alike Structural Re-parameterization module added after the bottleneck."""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()
        self.c = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(
            nn.Sequential(
                Conv(self.c, self.c, 3, 1, g=g),
                Conv(self.c, self.c, 3, 1, g=g)
            ) for _ in range(n)
        )
        # 🟢 THE ASR ATTENTION INJECTION 🟢
        self.att = GlobalSelfAttention(c2) # GlobalSelfAttention module

    def forward(self, x):
        # YOLOv8 C2f Logic
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        out = self.cv2(torch.cat(y, 1))

        # Apply GlobalSelfAttention on the output features
        return self.att(out)

print("✅ C2f_ASR block defined, integrating GlobalSelfAttention.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ C2f_ASR block defined, integrating GlobalSelfAttention.


In [ ]:
import torch
import torch.nn as nn
from ultralytics.nn.modules import Conv, C2f # Required for model inspection
from ultralytics import YOLO

# Re-using ASR_Attention and C2f_ASR definitions to ensure context for execution
class GlobalSelfAttention(nn.Module):
    def __init__(self, in_channels):
        super(GlobalSelfAttention, self).__init__()
        self.in_channels = in_channels

        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.final_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        combined_features = out1 + out2
        attention_map = self.sigmoid(self.final_conv(combined_features))
        out = x * attention_map
        return out

class C2f_ASR(nn.Module):
    """C2f block with Attention-alike Structural Re-parameterization module added after the bottleneck."""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()
        self.c = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(
            nn.Sequential(
                Conv(self.c, self.c, 3, 1, g=g),
                Conv(self.c, self.c, 3, 1, g=g)
            )
            for _ in range(n)
        )
        # 🟢 THE ASR ATTENTION INJECTION 🟢
        self.att = GlobalSelfAttention(c2) # GlobalSelfAttention module

    def forward(self, x):
        # YOLOv8 C2f Logic
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        out = self.cv2(torch.cat(y, 1))

        # Apply ASR Attention on the output features
        return self.att(out)


def create_asr_yolo_model(model_path='yolov8n.pt'):
    """
    Loads a standard YOLO model and replaces C2f blocks
    with C2f_ASR blocks in the backbone/head.
    """
    print(f"🏗 Loading base model: {model_path}...")
    model = YOLO(model_path)

    print("🔁 Injecting Attention-alike Structural Re-parameterization Modules (ASR) into architecture...")

    nn_model = model.model

    for name, module in nn_model.named_modules():
        if isinstance(module, C2f):
            parent_name = name.rsplit('.', 1)[0] if '.' in name else ''
            child_name = name.rsplit('.', 1)[-1]

            if parent_name:
                parent = nn_model.get_submodule(parent_name)
            else:
                parent = nn_model

            # Extract parameters from existing C2f
            c1 = module.cv1.conv.in_channels
            c2 = module.cv2.conv.out_channels
            n = len(module.m)

            # Create new ASR C2f
            new_module = C2f_ASR(c1, c2, n=n)

            # Replace
            setattr(parent, child_name, new_module)

    print("✅ Model transformation complete. ASR Attention active.")
    return model

# Usage: Create the ASR-enhanced model
model_asr = create_asr_yolo_model('yolov8n.pt')

print("✅ STEP 3 COMPLETED: ASR Attention Architecture ready.")

🏗 Loading base model: yolov8n.pt...
🔁 Injecting Attention-alike Structural Re-parameterization Modules (ASR) into architecture...
✅ Model transformation complete. ASR Attention active.
✅ STEP 3 COMPLETED: ASR Attention Architecture ready.


In [ ]:
# 1️⃣ Remove any previously downloaded dataset directory to prevent conflicts
import shutil

old_dataset_path = '/content/datasets/brain_tumor_detection'
if os.path.exists(old_dataset_path):
    shutil.rmtree(old_dataset_path)
    print(f"Cleaned up old dataset directory: {old_dataset_path}")

# 2️⃣ Download the new dataset from Kaggle
print("Downloading new dataset from Kaggle: sartajbhuvaji/brain-tumor-classification-mri...")
!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri

# 3️⃣ Unzip the dataset to a temporary location
print("Unzipping dataset...")
new_zip_file = 'brain-tumor-classification-mri.zip'
tmp_extract_dir = '/tmp/figshare_dataset_download'
os.makedirs(tmp_extract_dir, exist_ok=True)
!unzip -q {new_zip_file} -d {tmp_extract_dir}

# 4️⃣ Define the target directory for the organized dataset
final_dataset_root = '/content/datasets/figshare_mri'
os.makedirs(final_dataset_root, exist_ok=True)

# 5️⃣ Move contents from the temporary extraction folder to the final dataset directory
# The dataset typically unzips into a folder like 'brain_tumor_dataset' which contains Training, Testing, Validation
# We need to find this intermediate folder if it exists, or assume direct extraction.

extracted_subfolders = [f.name for f in os.scandir(tmp_extract_dir) if f.is_dir()]

# Corrected logic to move the subfolders found directly in tmp_extract_dir
if len(extracted_subfolders) > 0:
    for folder_name in extracted_subfolders:
        source_path = os.path.join(tmp_extract_dir, folder_name)
        destination_path = os.path.join(final_dataset_root, folder_name)
        shutil.move(source_path, destination_path)
    print(f"Moved content from {tmp_extract_dir} to {final_dataset_root}")
else:
    print(f"No subfolders found in {tmp_extract_dir} to move.")

# 6️⃣ Clean up the downloaded zip file and temporary extraction folder
!rm {new_zip_file}
!rm -r {tmp_extract_dir}

print(f"✅ New dataset downloaded, unzipped, and reorganized to {final_dataset_root}")

Dataset URL: https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
License(s): MIT
  0% 0.00/86.8M [00:00<?, ?B/s]
100% 86.8M/86.8M [00:00<00:00, 1.72GB/s]
Unzipping dataset...
Moved content from /tmp/figshare_dataset_download to /content/datasets/figshare_mri
✅ New dataset downloaded, unzipped, and reorganized to /content/datasets/figshare_mri


In [ ]:
import os
from pathlib import Path
from PIL import Image # Pillow for image size
import shutil # Import shutil for file operations
from sklearn.model_selection import train_test_split

print("🛠 Starting dataset conversion to YOLO format...")

source_dataset_root = '/content/datasets/figshare_mri' # Updated source path
target_dataset_root = '/content/datasets/figshare_mri_yolo' # Updated target path

# Class mapping for YOLO labels - UPDATED CLASS NAMES
class_names = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
class_to_id = {name: i for i, name in enumerate(class_names)}

# Create target directories
for split in ['train', 'val', 'test']:
    Path(f'{target_dataset_root}/{split}/images').mkdir(parents=True, exist_ok=True)
    Path(f'{target_dataset_root}/{split}/labels').mkdir(parents=True, exist_ok=True)

# Process Training split to create train and validation sets
training_source_path = Path(source_dataset_root) / 'Training'
if training_source_path.exists():
    print(f"Processing Training split for train/val...")
    all_training_images = []
    for class_folder in training_source_path.iterdir():
        if class_folder.is_dir():
            class_name = class_folder.name.lower()
            if class_to_id.get(class_name) is not None:
                all_training_images.extend(list(class_folder.glob('*.jpg')))

    # Split training images into train and validation
    train_images, val_images = train_test_split(all_training_images, test_size=0.2, random_state=42)

    # Copy images and create labels for the training set
    for img_file in train_images:
        class_name = img_file.parent.name.lower()
        class_id = class_to_id[class_name]
        shutil.copy(img_file, Path(target_dataset_root) / 'train' / 'images' / img_file.name)
        label_filename = img_file.stem + '.txt'
        with open(Path(target_dataset_root) / 'train' / 'labels' / label_filename, 'w') as f:
            f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")

    # Copy images and create labels for the validation set
    for img_file in val_images:
        class_name = img_file.parent.name.lower()
        class_id = class_to_id[class_name]
        shutil.copy(img_file, Path(target_dataset_root) / 'val' / 'images' / img_file.name)
        label_filename = img_file.stem + '.txt'
        with open(Path(target_dataset_root) / 'val' / 'labels' / label_filename, 'w') as f:
            f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")
else:
    print(f"⚠️  Warning: Original training folder '{training_source_path}' not found. No train/val split created.")

# Process Testing split for the test set
testing_source_path = Path(source_dataset_root) / 'Testing'
if testing_source_path.exists():
    print(f"Processing Testing split for test...")
    for class_folder in testing_source_path.iterdir():
        if class_folder.is_dir():
            class_name = class_folder.name.lower()
            class_id = class_to_id.get(class_name)

            if class_id is None:
                print(f"  Skipping unknown class folder: {class_name}")
                continue

            for img_file in class_folder.glob('*.jpg'):
                shutil.copy(img_file, Path(target_dataset_root) / 'test' / 'images' / img_file.name)
                label_filename = img_file.stem + '.txt'
                with open(Path(target_dataset_root) / 'test' / 'labels' / label_filename, 'w') as f:
                    f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")
else:
    print(f"⚠️  Warning: Original testing folder '{testing_source_path}' not found. No test split created.")

print(f"✅ Dataset conversion complete. YOLO formatted data is at {target_dataset_root}")

🛠 Starting dataset conversion to YOLO format...
Processing Training split for train/val...
Processing Testing split for test...
✅ Dataset conversion complete. YOLO formatted data is at /content/datasets/figshare_mri_yolo


In [ ]:
# 3️⃣ Create the data.yaml file
# This file tells YOLO where to find the images, labels, and class names.

# Define paths and class names - UPDATED FOR NEW DATASET
dataset_path = '/content/datasets/figshare_mri_yolo' # Updated path for YOLO formatted data

# Define the content for data.yaml
data_yaml_content = {
    'path': dataset_path, # Dataset root directory
    'train': 'train/images', # Relative path for training images
    'val': 'val/images',   # Relative path for validation images (even if empty, for consistency)
    'test': 'test/images', # Relative path for test images (optional, for final evaluation)
    'nc': 4,                  # Number of classes
    'names': ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']  # Updated Class names
}

# Create the data.yaml file
os.makedirs('/content/datasets', exist_ok=True) # Ensure the directory exists
with open('/content/datasets/data.yaml', 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print("✅ data.yaml created:")
with open('/content/datasets/data.yaml', 'r') as f:
    print(f.read())

print("✅ STEP 2 COMPLETED: Dataset prepared and data.yaml created.")

✅ data.yaml created:
names:
- glioma_tumor
- meningioma_tumor
- no_tumor
- pituitary_tumor
nc: 4
path: /content/datasets/figshare_mri_yolo
test: test/images
train: train/images
val: val/images

✅ STEP 2 COMPLETED: Dataset prepared and data.yaml created.


In [ ]:
print("🚀 Starting Training for Global Self-Attention-enhanced YOLOv8...")

# Check if data.yaml exists from Step 2
yaml_path = '/content/datasets/data.yaml'
if not os.path.exists(yaml_path):
    raise FileNotFoundError(f"❌ data.yaml not found at {yaml_path}. Did you run Step 2?")

# 2. Run Training
# Note: We do NOT pass 'model=' inside train(), because we are training the object itself.
results_asr = model_asr.train(
    data=yaml_path,
    epochs=50,             # Adjust based on time (50-100 is usually good for initial results)
    imgsz=640,             # Standard MRI resolution usually fits well in 640
    batch=16,              # Reduce to 8 if you run out of GPU memory
    patience=10,           # Stop early if no improvement for 10 epochs
    project='runs/train',  # Save results to /content/runs/train
    name='asr_yolo',       # Name of the specific run folder
    exist_ok=True,         # Overwrite existing folder if it exists
    optimizer='AdamW',     # AdamW often converges better for medical data than SGD
    lr0=0.001,             # Initial learning rate
    amp=False,             # Disable Mixed Precision to prevent errors with custom layers
    plots=True             # Generate training graphs automatically
)

print(f"✅ STEP 4 COMPLETED: Training finished for Global Self-Attention YOLOv8.")
print(f"   -> Best weights saved at: {results_asr.save_dir}/weights/best.pt")
print(f"   -> Training graphs saved at: {results_asr.save_dir}")

🚀 Starting Training for Global Self-Attention-enhanced YOLOv8...
Ultralytics 8.3.234 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=asr_yolo, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW,